In [12]:
#task1
from datetime import datetime
import requests, json

def collectWeather(cities):
    data=[]
    fname = datetime.strftime(datetime.today(), '%Y-%m-%d_%H:%M.json')
    print(f"Preparing {fname}")
    tgtPath="./raw_files" #"/app/raw_files"
    
    for city in cities :
        req = f"https://api.openweathermap.org/data/2.5/weather?q={city},fr&APPID=961373402b6adffd1ce1e7d1e6ce1682"
        r = requests.get(req)
        data.append({"city": city, "infos":r.json()})
        print(city,r.json(),"\n")
    print(data)

    with open(f"{tgtPath}/{fname}","w", encoding="utf_8") as f: 
        json.dump(data,f,indent=4)

    f.close()   


collectWeather( ['paris, France', 'Paris, Kentucky ', 'london', 'Washington, District of Columbia'])

Preparing 2026-05-20_23:53.json
paris, France {'coord': {'lon': 2.3488, 'lat': 48.8534}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'base': 'stations', 'main': {'temp': 285.16, 'feels_like': 284.87, 'temp_min': 284.29, 'temp_max': 287.58, 'pressure': 1028, 'humidity': 94, 'sea_level': 1028, 'grnd_level': 1018}, 'visibility': 10000, 'wind': {'speed': 1.03, 'deg': 240}, 'clouds': {'all': 0}, 'dt': 1779321060, 'sys': {'type': 1, 'id': 6550, 'country': 'FR', 'sunrise': 1779336116, 'sunset': 1779391963}, 'timezone': 7200, 'id': 2988507, 'name': 'Paris', 'cod': 200} 

Paris, Kentucky  {'coord': {'lon': 2.3488, 'lat': 48.8534}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'base': 'stations', 'main': {'temp': 285.04, 'feels_like': 284.74, 'temp_min': 283.92, 'temp_max': 286.6, 'pressure': 1028, 'humidity': 94, 'sea_level': 1028, 'grnd_level': 1018}, 'visibility': 10000, 'wind': {'speed': 1.03, 'deg': 240}, 'clo

In [13]:
#task2
import os
import pandas as pd
import json

def transform_data_into_csv(n_files=None, filename='data.csv'):
    main_path="." #resp /app
    parent_folder = f'{main_path}/raw_files'
    files = sorted(os.listdir(parent_folder), reverse=True)

    if n_files:
        files = files[:n_files]

    dfs = []
    for f in files:
        with open(os.path.join(parent_folder, f), 'r') as file:
            data_temp = json.load(file)

        for data_city in data_temp:
            print(data_city)
            dfs.append(
                {
                    'temperature': data_city['infos']['main']['temp'],
                    'city': data_city['city'],
                    'pression': data_city['infos']['main']['pressure'],
                    'date': f.split('.')[0].replace("_", " ")
                }
            )
    df = pd.DataFrame(dfs)
    print('\n', df.head(10))
    df.to_csv(os.path.join(f'{main_path}/clean_data', filename), index=False)

transform_data_into_csv()
transform_data_into_csv(n_files=20)
transform_data_into_csv(filename="fulldata.csv")


{'city': 'paris, France', 'infos': {'coord': {'lon': 2.3488, 'lat': 48.8534}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'base': 'stations', 'main': {'temp': 285.16, 'feels_like': 284.87, 'temp_min': 284.29, 'temp_max': 287.58, 'pressure': 1028, 'humidity': 94, 'sea_level': 1028, 'grnd_level': 1018}, 'visibility': 10000, 'wind': {'speed': 1.03, 'deg': 240}, 'clouds': {'all': 0}, 'dt': 1779321060, 'sys': {'type': 1, 'id': 6550, 'country': 'FR', 'sunrise': 1779336116, 'sunset': 1779391963}, 'timezone': 7200, 'id': 2988507, 'name': 'Paris', 'cod': 200}}
{'city': 'Paris, Kentucky ', 'infos': {'coord': {'lon': 2.3488, 'lat': 48.8534}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'base': 'stations', 'main': {'temp': 285.04, 'feels_like': 284.74, 'temp_min': 283.92, 'temp_max': 286.6, 'pressure': 1028, 'humidity': 94, 'sea_level': 1028, 'grnd_level': 1018}, 'visibility': 10000, 'wind': {'speed': 1.03, 'deg': 2

In [ ]:
#task3

In [14]:
#task4'
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from joblib import dump

def compute_model_score(model, X, y):
    # computing cross val
    cross_validation = cross_val_score(
        model,
        X,
        y,
        cv=3,
        scoring='neg_mean_squared_error')

    model_score = cross_validation.mean()
    return model_score
	
def train_and_save_model(model, X, y, path_to_model='./app/model.pckl'):
    # training the model
    model.fit(X, y)
    # saving model
    print(str(model), 'saved at ', path_to_model)
    dump(model, path_to_model)

def prepare_data(path_to_data='/app/clean_data/fulldata.csv'):
    # reading data
    df = pd.read_csv(path_to_data)
    # ordering data according to city and date
    df = df.sort_values(['city', 'date'], ascending=True)

    dfs = []
    for c in df['city'].unique():
        df_temp = df[df['city'] == c]
		# creating target
        df_temp.loc[:, 'target'] = df_temp['temperature'].shift(1)
		# creating features
        for i in range(1, 10):
            df_temp.loc[:, 'temp_m-{}'.format(i)] = df_temp['temperature'].shift(-i)
    		# deleting null values
            df_temp = df_temp.dropna()
            dfs.append(df_temp)
	
	    # concatenating datasets
        df_final = pd.concat(dfs,axis=0,ignore_index=False)

	# deleting date variable
    df_final = df_final.drop(['date'], axis=1)
	# creating dummies for city variable
    df_final = pd.get_dummies(df_final)
    features = df_final.drop(['target'], axis=1)
    target = df_final['target']
    return features, target


In [21]:
#task4''

import numpy as np
X, y = prepare_data('./clean_data/fulldata.csv')
X=X.fillna(0)
print(X,y)

scores=[]

#sequential execution is ok for small models
models=[LinearRegression(),DecisionTreeRegressor(),RandomForestRegressor()]
for m,model in enumerate(models):
    scores.append(compute_model_score(model, X, y))

#to translate into dags with XCom
print(scores)
bestModel=models[scores.index(min(scores))]
# using neg_mean_square_error and find the min of scores
train_and_save_model(bestModel, X, y,'./clean_data/best_model.pickle' )




    temperature  pression  temp_m-1  temp_m-2  temp_m-3  temp_m-4  temp_m-5  \
21       285.55      1028    285.55    285.55       0.0       0.0       0.0   
17       285.55      1028    285.55    285.52       0.0       0.0       0.0   
13       285.55      1028    285.52    285.20       0.0       0.0       0.0   
9        285.52      1028    285.20      0.00       0.0       0.0       0.0   
5        285.20      1028    285.04      0.00       0.0       0.0       0.0   
21       285.55      1028    285.55    285.55       0.0       0.0       0.0   
17       285.55      1028    285.55    285.52       0.0       0.0       0.0   
13       285.55      1028    285.52    285.20       0.0       0.0       0.0   
23       296.74      1015    296.74    296.74       0.0       0.0       0.0   
19       296.74      1015    296.74    296.81       0.0       0.0       0.0   
15       296.74      1015    296.81    296.90       0.0       0.0       0.0   
11       296.81      1015    296.90      0.00       

In [ ]:
#task4'''


In [ ]:
task5